# 14.18 怎樣公平比較堆疊與循環模型？


一個120萬參數模型多跑五輪，贏過一個120萬參數只跑一輪的模型，可以說「同參數下較準」。但不能因此說「同計算下較準」；新模型花了不同工作量。比較前要先決定要回答哪個問題。

**固定參數**的比較，適合參數儲存受限的情境，允許計算不同；**固定工作量**的比較，則盡量對齊每題執行的區塊數或估算 FLOPs。FLOPs 是浮點運算次數的計量，不是秒數；共享會改記憶體讀取與裝置利用率，所以最後仍應量測時間。若參數與工作都改了，要完整列出，不能把所有收益歸到共享。

沿用[14.11](https://birdhackor.github.io/tiny-perceptron-vlm/14.11.html)的假設數字（字的embedding表與輸出層20萬、每個區塊100萬參數），把三種設計的參數與每題區塊執行次數列出來：


In [ ]:
# @title 準備本節的工具（首次執行）
from pathlib import Path
import os
import subprocess
import sys

# Colab 會先下載專案；本機 Jupyter 沿用已開啟的 repo。
try:
    import google.colab
except ImportError:
    in_colab = False
else:
    in_colab = True

if in_colab:
    root = Path("/content/tiny-perceptron-vlm")
    if not root.exists():
        subprocess.run(
            ["git", "clone", "--depth", "1", "https://github.com/birdhackor/tiny-perceptron-vlm.git", str(root)],
            env={**os.environ, "GIT_LFS_SKIP_SMUDGE": "1"},
            check=True,
        )
    subprocess.run([sys.executable, "-m", "pip", "install", "--quiet", "-e", str(root)], check=True)
    os.chdir(root)

root = Path.cwd().resolve()
while not (root / "pyproject.toml").exists() and root != root.parent:
    root = root.parent
if not (root / "tiny_perceptron").is_dir():
    raise RuntimeError("本機請先依 course/first-steps.md 開啟 repo；免安裝練習可使用本節的 Colab 入口")
sys.path.insert(0, str(root))
import torch

torch.set_num_threads(1)
torch.manual_seed(42)

In [ ]:
other, block = 200_000, 1_000_000
designs = [("共享一個區塊跑一輪", 1, 1), ("共享一個區塊跑五輪", 1, 5), ("堆疊五個區塊", 5, 5)]
for name, copies, rounds in designs:
    print(name, "參數", other + copies * block, "每題區塊執行", rounds, "次")

`200_000` 的底線只是分位，等於200000。`designs` 每一項是「名稱、區塊參數份數、每題執行幾輪」，`for name, copies, rounds in designs:` 每次取出一項的三個值。前兩列參數都是1,200,000，只差執行次數：比較它們是固定參數。後兩列每題都執行五次區塊，參數卻是1,200,000對5,200,000：比較它們才接近固定工作量。開頭那句「同參數下較準」比的是前兩列，不能拿來回答後兩列的問題。

做對照（control）實驗還要固定：相同的家族切分、tokenizer、輸入長度、訓練工作與挑選檢查點的規則。隨機種子影響結果；少量種子的短實驗只能支持該次配方，不能推廣成架構排名。若已有訓練好的 MoE 助理，拿來和 Looped 解題器比較用途很有價值，但兩者資料和訓練歷史不同，就不是區塊共用參數的因果實驗。

輪數曲線每一點至少記下四種量：驗證或測試資料上的交叉熵、最後答案正確率、實際推論時間、峰值記憶體。每題都用相同的評估材料；在相同裝置與每批同時處理題數（batch size）下先暖機，再多次量測。GPU 收到工作後會先排進佇列、稍後才真正計算，所以計時前後都要等它算完（同步），否則碼錶只量到把工作交出去的時間；第16章（[16.1](https://birdhackor.github.io/tiny-perceptron-vlm/16.1.html)）再說明量法。上面的程式只數參數與次數，小CPU機制示範也不能當成GPU加速證據。

| 要問的問題 | 可以固定什麼 | 還須公開什麼 |
| --- | --- | --- |
| 有限記憶體能不能多做工作？ | 參數量 | 輪數、時間、品質 |
| 同工作量哪種安排較好？ | 區塊執行／估算FLOPs | 參數、訓練條件、實際秒數 |
| 產品該選哪個檔位？ | 凍結產品與題目 | 每輪品質、總成本、失敗例 |

這些記錄讓第三輪最好、偶數輪變差或外推失敗都成為可解釋結果。若結果沒有支持多輪收益，機制教學仍然成立；要標成可用的成品，則須另外通過第19章預先訂的能力門檻。兩種驗收不能互相替代。

練習：兩個模型更新次數相同，但一個平均展開一輪、另一個五輪，能否稱訓練預算相同？至少不能只憑更新次數下結論。先記每次做了多少工作，再依研究問題決定要如何配對；不要為了湊出整齊表格隱藏代價。也可以在程式的 `designs` 加一項 `("堆疊兩個區塊", 2, 2)`，先預測它的參數2,200,000、每題執行2次，再判斷它能和哪一列做固定參數或固定工作量的比較：都不行，它的參數和次數都和另外三列不同。
